In [1]:
import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D

**데이터 준비 및 전처리**

In [2]:
BATCH_SIZE = 128
NUM_CLASSES = 10
NUM_EPOCHS = 2

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

y_train = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test = keras.utils.to_categorical(y_test, NUM_CLASSES)

**CNN(합성곱 신경망) 모델 구조 설계**

In [3]:
model = Sequential()
from keras.layers import Input
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(32, kernel_size = (3, 3), activation='relu'))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))

model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(NUM_CLASSES, activation='softmax'))

**모델 컴파일 / 학습 진행**

In [4]:
model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

print("CNN 모델 학습을 시작합니다...")

model.fit(x_train, y_train, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS, verbose=1)

CNN 모델 학습을 시작합니다...
Epoch 1/2
469/469 ━━━━━━━━━━━━━━━━━━━━ 40s 76ms/step - accuracy: 0.9254 - loss: 0.2432
Epoch 2/2
469/469 ━━━━━━━━━━━━━━━━━━━━ 37s 78ms/step - accuracy: 0.9742 - loss: 0.0872


**모델 성능 평가**

In [5]:
cnn_score = model.evaluate(x_test, y_test, verbose=0)
print('최종 테스트 정확도(Accuracy): %.2f%%' % (cnn_score[1] * 100))

최종 테스트 정확도(Accuracy): 98.80%


**그림판으로 구현(추가)**

In [ ]:
from IPython.display import HTML, display, clear_output
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import matplotlib.pyplot as plt
import time
import tensorflow as tf

def continuous_canvas(model):
    while True:
        uid = int(time.time() * 1000)

        canvas_html = f"""
        <div style="display:flex; flex-direction:column; align-items:center; border: 2px solid #ddd; padding: 20px; border-radius: 10px; width: max-content;">
            <h3 style="margin-top:0;">마우스로 숫자를 그려보세요 (0~9)</h3>
            <canvas id="canvas_{uid}" width="280" height="280" style="border: 1px solid black; background-color: black; cursor: crosshair; touch-action: none;"></canvas>
            <div style="margin-top: 15px;">
                <button id="clear_btn_{uid}" style="padding: 10px 20px; margin-right: 10px; font-weight: bold;">지우기</button>
                <button id="predict_btn_{uid}" style="padding: 10px 20px; margin-right: 10px; font-weight: bold; color: white; background-color: #4285F4; border: none; border-radius: 5px;">예측하기</button>
                <button id="exit_btn_{uid}" style="padding: 10px 20px; font-weight: bold; color: white; background-color: #EA4335; border: none; border-radius: 5px;">종료</button>
            </div>
        </div>
        <script>
        var canvas = document.getElementById('canvas_{uid}');
        var ctx = canvas.getContext('2d');
        ctx.lineWidth = 15;
        ctx.lineCap = 'round';
        ctx.lineJoin = 'round';
        ctx.strokeStyle = 'white';

        var drawing = false;
        var pos = {{ x: 0, y: 0 }};

        function setPosition(e) {{
            var rect = canvas.getBoundingClientRect();
            pos.x = (e.clientX || e.touches[0].clientX) - rect.left;
            pos.y = (e.clientY || e.touches[0].clientY) - rect.top;
        }}

        function draw(e) {{
            if (!drawing) return;
            e.preventDefault();
            ctx.beginPath();
            ctx.moveTo(pos.x, pos.y);
            setPosition(e);
            ctx.lineTo(pos.x, pos.y);
            ctx.stroke();
        }}

        canvas.addEventListener('mousedown', function(e) {{ drawing = true; setPosition(e); }});
        canvas.addEventListener('mousemove', draw);
        canvas.addEventListener('mouseup', function(e) {{ drawing = false; }});
        canvas.addEventListener('mouseout', function(e) {{ drawing = false; }});

        canvas.addEventListener('touchstart', function(e) {{ drawing = true; setPosition(e); }}, {{passive: false}});
        canvas.addEventListener('touchmove', draw, {{passive: false}});
        canvas.addEventListener('touchend', function(e) {{ drawing = false; }});

        document.getElementById('clear_btn_{uid}').addEventListener('click', function() {{
            ctx.clearRect(0, 0, canvas.width, canvas.height);
        }});

        var promise_{uid} = new Promise(resolve => {{
            document.getElementById('predict_btn_{uid}').onclick = () => resolve(canvas.toDataURL('image/png'));
            document.getElementById('exit_btn_{uid}').onclick = () => resolve("EXIT");
        }});
        </script>
        """

        clear_output(wait=True)
        display(HTML(canvas_html))

        data = eval_js(f'promise_{uid}')

        if data == "EXIT":
            clear_output(wait=True)
            break

        binary = b64decode(data.split(',')[1])
        img_array = np.frombuffer(binary, dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

        if img is None:
            continue

        # Find all non-zero pixels (drawn content)
        coords = np.argwhere(img > 0)
        if len(coords) == 0:
            # If nothing is drawn, continue to the next iteration
            continue

        # Get bounding box coordinates of the drawn content
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)

        # Calculate center and max dimension of the drawn content
        center_x = (x_min + x_max) // 2
        center_y = (y_min + y_max) // 2
        max_content_dim = max(x_max - x_min, y_max - y_min)

        # Define a scale factor to add padding around the digit, typically MNIST digits are ~20x20 in a 28x28 canvas.
        # A scale_factor of 1.2 ensures some margin around the digit.
        scale_factor = 1.2
        target_square_dim_in_original_scale = int(max_content_dim * scale_factor)

        # Ensure the target square dimension is not too small or too large
        target_square_dim_in_original_scale = max(target_square_dim_in_original_scale, max_content_dim)
        target_square_dim_in_original_scale = min(target_square_dim_in_original_scale, img.shape[0]) # Can't exceed 280 (canvas size)

        # Calculate the padded square region in the original 280x280 canvas, centered on the digit
        half_side = target_square_dim_in_original_scale // 2

        # Calculate the coordinates for cropping, centered around the digit
        crop_x_min = max(0, center_x - half_side)
        crop_y_min = max(0, center_y - half_side)
        crop_x_max = min(img.shape[1], center_x + half_side)
        crop_y_max = min(img.shape[0], center_y + half_side)

        # Adjust for clipping: ensure the final cropped region is square
        current_crop_width = crop_x_max - crop_x_min
        current_crop_height = crop_y_max - crop_y_min

        final_crop_dim = min(current_crop_width, current_crop_height)

        # Recalculate mins/maxs to ensure a perfect square of final_crop_dim, centered within the previously clipped region
        if current_crop_width > final_crop_dim:
            diff_x = current_crop_width - final_crop_dim
            crop_x_min += diff_x // 2
            crop_x_max -= (diff_x - diff_x // 2)
        if current_crop_height > final_crop_dim:
            diff_y = current_crop_height - final_crop_dim
            crop_y_min += diff_y // 2
            crop_y_max -= (diff_y - diff_y // 2)

        # Crop the image based on the adjusted square region
        img_cropped = img[crop_y_min:crop_y_max, crop_x_min:crop_x_max]

        # Resize this cropped, padded square image to 20x20
        img_scaled_to_20x20 = cv2.resize(img_cropped, (20, 20), interpolation=cv2.INTER_AREA)

        # Create a new 28x28 black canvas
        final_img_28x28 = np.zeros((28, 28), dtype=np.uint8)

        # Paste the 20x20 image into the center of the 28x28 canvas (this adds the 4-pixel border)
        final_img_28x28[4:24, 4:24] = img_scaled_to_20x20

        # Use this final 28x28 image for prediction
        img_resized = final_img_28x28

        input_array = img_resized.astype(np.float32).reshape(1, 28, 28, 1) / 255.0
        input_tensor = tf.convert_to_tensor(input_array)

        prediction = model(input_tensor, training=False).numpy()
        predicted_number = np.argmax(prediction)

        plt.figure(figsize=(3, 3))
        plt.imshow(img_resized, cmap='gray')
        plt.title(f"AI Prediction: {predicted_number}", fontsize=18, color='blue')
        plt.axis('off')
        plt.show()

        print(f"인공지능 예측 결과: {predicted_number}")
        print(f"각 숫자별 확률: \n{np.round(prediction[0] * 100, 2)}%\n")

        next_html = f"""
        <div style="margin-top: 10px;">
            <button id="next_btn_{uid}" style="padding: 10px 20px; font-weight: bold; background-color: #34A853; color: white; border: none; border-radius: 5px; margin-right: 10px;">새 캔버스 열기</button>
            <button id="exit2_btn_{uid}" style="padding: 10px 20px; font-weight: bold; background-color: #EA4335; color: white; border: none; border-radius: 5px;">테스트 종료</button>
        </div>
        <script>
        var next_p_{uid} = new Promise(resolve => {{
            document.getElementById('next_btn_{uid}').onclick = () => resolve("NEXT");
            document.getElementById('exit2_btn_{uid}').onclick = () => resolve("EXIT");
        }});
        </script>
        """
        display(HTML(next_html))

        action = eval_js(f'next_p_{uid}')

        if action == "EXIT":
            clear_output(wait=True)
            break

continuous_canvas(model)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Display 25 random images from the training data
plt.figure(figsize=(10, 10))
for i in range(25):
    idx = np.random.randint(0, len(x_train))
    plt.subplot(5, 5, i+1)
    plt.imshow(x_train[idx].reshape(28, 28), cmap='gray')
    plt.title(f'Label: {np.argmax(y_train[idx])}')
    plt.axis('off')
plt.suptitle('Training Data Samples', fontsize=16)
plt.show()